# Cinema Booking System  
## Documentation Report & Code 

---

**Student Name:** Aish Waheed  
**UWE ID:** 26031834  
**Villa ID:** S2610767  
**Module:** PRINCIPLES OF PROGRAMMING  
**Date:** 29 April 2026  

---

## Introduction

This Python program is a CLI-based cinema booking system. It allows users to view movies, book tickets, and sort movie listings. It also provides an admin mode where an administrator can add new movies, remove existing movies, view all movies, and sort the movie list.

The program uses a list of dictionaries to store movie data. Each movie contains details such as ID, title, price, date, and number of bookings. The system also uses helper functions to validate user input and menu functions to control the flow of the program.

---

## Code

### Import and Global Variables

This block imports the `datetime` class from Python’s datetime module. It is used to create and format movie dates.

The `MAX_BOOKINGS` variable stores the maximum number of bookings allowed for each movie. In this program, each movie can only have 5 bookings.

The `PASSWORD` variable stores the admin password. This is used to protect admin features such as adding or removing movies. The default password is set to `Welcome@123`.

The `last_movie_id` variable stores the most recent movie ID. When a new movie is added, this value is increased by 1 so that each new movie gets a unique ID.

In [ ]:
from datetime import datetime

MAX_BOOKINGS = 5
PASSWORD = "Welcome@123"
last_movie_id = 1005

### Movies List

This block creates the main list of movies used by the program. The `movies` variable is a list, and each item inside the list is a dictionary.

Each dictionary stores information about one movie. The `id` is used to uniquely identify the movie. The `title` stores the movie name. The `price` stores the ticket price. The `date` stores the movie date using a `datetime` object. The `bookings` value stores how many tickets have already been booked for that movie.

This list acts like a simple temporary database. Since the data is stored in memory, changes made while the program is running will be lost when the program closes.

In [ ]:
movies = [
    {
        "id": 1000,
        "title": "Inception",
        "price": 10.0,
        "date": datetime(2026, 5, 1),
        "bookings": 0,
    },
    {
        "id": 1001,
        "title": "The Dark Knight",
        "price": 12.5,
        "date": datetime(2026, 5, 2),
        "bookings": 0,
    },
    {
        "id": 1002,
        "title": "Interstellar",
        "price": 11.0,
        "date": datetime(2026, 5, 3),
        "bookings": 5,
    },
    {
        "id": 1003,
        "title": "Avengers: Endgame",
        "price": 13.0,
        "date": datetime(2026, 5, 4),
        "bookings": 5,
    },
    {
        "id": 1004,
        "title": "Spider-Man: No Way Home",
        "price": 12.0,
        "date": datetime(2026, 5, 5),
        "bookings": 0,
    },
]

### Input Helper Functions
The input helper functions are used to standardize how user input is collected and validated throughout the program. They ensure that only valid data is accepted by repeatedly prompting the user until correct input is provided. This helps prevent runtime errors and keeps the main program logic clean and easy to manage.

#### 1. `prompt_confirm()` Function
This function asks the user to confirm an action by entering Y or N.

The function receives a `message` as a parameter. That message is displayed to the user. The user’s input is cleaned using `.strip()` to remove extra spaces and `.lower()` to make the input lowercase.

If the user enters `y`, the function returns `True`. For any other input, it returns `False`.

This function is useful for actions that need confirmation, such as deleting a movie.

In [ ]:
def prompt_confirm(message):
    input_confirm = input(f"\n{message} ([Y]es / [N]o) => ")
    if input_confirm.strip().lower() == "y":
        return True
    else:
        return False

#### 2. `prompt_choice()` Function
This function is used when the program expects the user to enter a number, such as a menu option or movie ID.

The function keeps running inside a `while True` loop until the user enters a valid integer. The user input is first collected as text, then converted to an integer using `int()`.

If the conversion works, the number is returned. If the user enters something invalid, such as letters, Python raises a `ValueError`. The `except` block catches this error and displays an error message instead of crashing the program.

In [ ]:
def prompt_choice(message):
    while True:
        choice = input(f"\n{message}")
        try:
            choice = int(choice.strip())
            return choice
        except ValueError:
            print("Choose a valid option")

#### 3. `prompt_date()` Function
This function asks the user to enter a movie date.

The expected format is `DD-MM-YYYY`, for example `01-05-202`6. The function uses `datetime.strptime()` to convert the user’s text input into a real `datetime` object.

If the user enters the date in the correct format, the converted date is returned. If the format is wrong, the program shows an error message and asks again.

This function helps make sure all movie dates are stored consistently.

In [ ]:
def prompt_date():
    while True:
        input_date = input("Enter movie date (DD-MM-YYYY) => ")
        try:
            input_date = datetime.strptime(input_date.strip(), "%d-%m-%Y")
            return input_date
        except ValueError:
            print("Input a valid date in the format DD-MM-YYYY")

#### 4. `prompt_title()` Function
This function is used to get a valid movie title from the user.

The input is cleaned using `.strip()`, which removes spaces at the beginning and end. The condition `if not input_title` checks whether the title is empty.

If the user enters nothing or only spaces, an error message is shown. Otherwise, the valid title is returned.

This prevents blank movie titles from being added to the system.

In [ ]:
def prompt_title():
    while True:
        input_title = input("Enter the Movie title => ").strip()
        if not input_title:
            print("Title can't be blank or only whitespaces")
        else:
            return input_title

#### 5. `prompt_float()` Function
This function is used when the program needs a decimal number, such as a movie price or payment amount.

The function receives a custom message as a parameter. This makes it reusable in different places.

The user’s input is converted to a float using `float()`. If the input is valid, the number is returned. If the input is invalid, the program shows an error message and asks again.

In [ ]:
def prompt_float(message):
    while True:
        input_float = input(message)
        try:
            input_float = float(input_float.strip())
            return input_float
        except ValueError:
            print("Enter a valid value")

### `run_menu()` Function
This is one of the most important functions in the program. It creates a reusable menu system.

The function receives two parameters: `menu_list` and `menu_title`. The `menu_list` contains menu options, and the `menu_title` is displayed at the top of the menu.

The function loops through the menu list using `enumerate()` and prints each option with a number. It also adds option `0` for going back.

After the user chooses an option, the program checks whether the choice is valid. If the user enters `0`, the loop breaks and the menu closes. If the choice is outside the valid range, an error message is shown.

If the choice is valid, the matching function is called using:

```python
menu_list[choice - 1]["action"]()
```

The reason for `choice - 1` is that list indexes start from `0`, but menu options start from `1`.

In [ ]:
def run_menu(menu_list, menu_title):
    while True:
        print("=" * 35)
        print(f" {menu_title}")
        print("=" * 35)
        for i, item in enumerate(menu_list, start=1):
            print(f" {i}. {item['entry_name']}")
        print(" 0. Back")
        choice = prompt_choice(" Choose an option => ")

        if choice == 0:
            break

        if choice < 0 or choice > len(menu_list):
            print("Enter a valid choice")
            continue
        print("_" * 35)
        menu_list[choice - 1]["action"]()

### `add_movie()` Function
This function allows the admin to add a new movie.

It creates a new dictionary called `new_movie`. The `title`, `price`, and `date` are collected using helper functions. The number of `bookings` is set to `0` because a newly added movie has not been booked yet.

The function uses `global last_movie_id` because it needs to update the global `last_movie_id` variable. The ID is increased by `1` and assigned to the new movie.

Finally, the new movie dictionary is added to the `movies` list using `.append()`.

In [ ]:
def add_movie():
    new_movie = {
        "title": prompt_title(),
        "price": prompt_float("Enter price => "),
        "date": prompt_date(),
        "bookings": 0,
    }
    global last_movie_id
    last_movie_id += 1
    new_movie["id"] = last_movie_id
    movies.append(new_movie)

### `list_movies()` Function

```python
movies_list = (
    movies if show_booked else [x for x in movies if x["bookings"] < MAX_BOOKINGS]
)
```

This function displays the movie records in a formatted table. The parameter `show_booked` controls whether all movies are shown or only movies that are still available for booking.

The most important part of this function is the line that creates `movies_list`. This line uses a **conditional expression** and a **list comprehension**.

A **conditional expression** is a shorter way of writing an `if...else` decision in one line. In this code, it means:

```python
if show_booked:
    movies_list = movies
else:
    movies_list = [x for x in movies if x["bookings"] < MAX_BOOKINGS]
```

The second part uses a **list comprehension**:

```python
[x for x in movies if x["bookings"] < MAX_BOOKINGS]
```

A list comprehension is a compact way to create a new list from an existing list. Here, it loops through every movie `x` in the `movies` list and only includes the movie if its bookings are less than `MAX_BOOKINGS`.

This is used because the user mode should not show movies that are already fully booked. Instead of manually creating an empty list and appending available movies one by one, the list comprehension makes the filtering shorter and clearer.

In [ ]:
def list_movies(show_booked=True):
    print(
        f"  {'Sn':>4.3} | {'ID':<6} | {'Title':<26.25} | {'Price':>7.6} | {'Date':<13} | {'Bookings':<9.8}"
    )
    print("=" * 85)

    """ This code checks if the argument show_booked is true. If it is true then the movies_list will reference the global
    movies variable. 
    If it is false, then a new list is generated filtering out all movies where bookings value exceeds or
    is equal to the global MAX_BOOKINGS variable. """
    movies_list = (
        movies if show_booked else [x for x in movies if x["bookings"] < MAX_BOOKINGS]
    )

    for index, item in enumerate(movies_list, start=1):
        formatted_date = datetime.strftime(item["date"], "%d-%b-%y")
        print(
            f"  {index:>4} | {item['id']:<6} | {item['title']:<26.25} | {item['price']:>7.2f}  | {formatted_date:<13} | {f'{item["bookings"]}/{MAX_BOOKINGS}':<9}"
        )
    print("\n" + "_" * 35 + " End of list " + "_" * 35)

### `select_movie()` Function

```python
selected_movie = next(
    ((i, m) for i, m in enumerate(movies) if m["id"] == choice), None
)
```

This function allows the user to select a movie by entering its movie ID. It first displays the movie list, then repeatedly asks the user for an ID until a valid movie is found or the user cancels.

The most complex line in this function uses a **generator expression** together with the `next()` function:

```python
((i, m) for i, m in enumerate(movies) if m["id"] == choice)
```

A **generator expression** looks similar to a list comprehension, but it does not create a full list in memory. Instead, it produces values one at a time.

Here, the generator loops through the `movies` list using `enumerate()`. The `enumerate()` function gives two values for each movie:

```python
i
```

is the index of the movie in the list, and

```python
m
```

is the movie dictionary itself.

The condition:

```python
if m["id"] == choice
```

means the generator only produces a result when the movie ID matches the user’s input.

The `next()` function then takes the first matching result from the generator. If no matching movie is found, the second argument `None` is returned instead.

This is useful because the program only needs the first matching movie. A generator expression is efficient here because it stops searching once a match is found instead of building a full list of results.

In [ ]:
def select_movie():
    print("\n=== Select Movie ===\n")
    list_movies()
    print("\n0. Cancel Selection")
    while True:
        choice = prompt_choice("\nEnter ID of movie to select => ")
        if choice == 0:
            return None
        selected_movie = next(
            ((i, m) for i, m in enumerate(movies) if m["id"] == choice), None
        )
        if selected_movie is None:
            print(f"Movie of ID:[{choice}] not found. Try again.")
            continue
        return selected_movie


### `remove_movie()` Function

This function allows the admin to remove a movie from the list.

It first calls `select_movie()` so the admin can choose which movie to remove. If the admin cancels the selection, the function stops.

The function then asks for confirmation before deleting the movie. If the selected movie already has bookings, it gives an extra warning. This is important because deleting a booked movie could affect users.

If the admin confirms, the movie is removed from the `movies` list using:

```python
movies.pop(selected_movie[0])
```

Here, `selected_movie[0]` is the index of the selected movie.

In [ ]:
def remove_movie():
    selected_movie = select_movie()
    if selected_movie is None:
        return
    confirmed = prompt_confirm(
        f'Are you sure you want to delete "{selected_movie[1]["title"]}" ?'
    )
    if selected_movie[1]["bookings"] > 0 and confirmed:
        confirmed = prompt_confirm(
            f"The movie you are about to delete has been booked {selected_movie[1]['bookings']} times.\nDo you still want to proceed ?"
        )
    if confirmed:
        movies.pop(selected_movie[0])
    else:
        print("Cancelled")

### `view_all()` Function
This is a simple wrapper function used to display movies with a heading.

It calls `list_movies()` and passes the `show_booked` value to it. This makes the function flexible because it can show all movies or only available movies depending on how it is called.

In [ ]:
def view_all(show_booked=True):
    print("\n=== All Movies ===\n")
    list_movies(show_booked)

### `sort_movies_list()` Function
```python
movies.sort(key=lambda m: m[key], reverse=desc)
```

This function sorts the movie list by a chosen field, such as title, date, or price.

The complex part of this function is the use of a **lambda function** inside the `.sort()` method.

A **lambda function** is a short anonymous function. It is useful when a small function is needed only once.

```python
lambda m: m[key]
```

means:

```python
def get_sort_value(m):
    return m[key]
```

For each movie dictionary `m`, the lambda returns the value stored under the selected key. For example, if `key` is `"price"`, the lambda returns each movie’s price. Python then uses those values to sort the list.

The `reverse=desc` part controls whether the sorting is ascending or descending. If `desc` is `False`, the list is sorted from lowest to highest or A to Z. If `desc` is `True`, the list is sorted in reverse order.

This approach is useful because one function can sort by different fields without writing separate sorting functions for title, price, and date.

In [ ]:
def sort_movies_list(key, desc=False):
    movies.sort(key=lambda m: m[key], reverse=desc)
    print("Movies sorted")
    view_all()

### `payment_confirmation()` Function
This function simulates payment confirmation.

It gets the actual ticket price from the selected movie using the movie index.

```python
actual_amount = movies[movie_index]["price"]
```

Then it asks the user to enter the payment amount. If the entered amount is not exactly equal to the movie price, the payment fails and the function returns `False`.

If the amount matches the price, the payment is successful and the function returns `True`.

This function is used before confirming a ticket booking.

In [ ]:
def payment_confirmation(movie_index):
    actual_amount = movies[movie_index]["price"]
    payment_amount = prompt_float("Enter Payment amount to confirm => ")
    if payment_amount != actual_amount:
        print("Payment Failed")
        return False
    else:
        print("Payment Successful")
        return True


### `book_ticket()` Function
This function handles the full ticket booking process.

First, it asks the user to select a movie. If the user cancels, the function stops.

Then it checks whether the selected movie still has available booking spaces:

```python
if selected[1]["bookings"] < MAX_BOOKINGS:
```

If the movie is fully booked, the user is asked to choose another movie.

Once an available movie is selected, the function calls `payment_confirmation()`. If payment fails, the booking is cancelled.

If payment succeeds, the movie’s booking count is increased by 1:

```python
selected[1]["bookings"] += 1
```

Finally, the program displays a message confirming that the ticket has been booked.

In [ ]:
def book_ticket():
    while True:
        selected = select_movie()
        if selected is None:
            return
        if selected[1]["bookings"] < MAX_BOOKINGS:
            break
        print("This movie has been fully booked. Try another movie.")
    payment_confirmed = payment_confirmation(selected[0])
    if not payment_confirmed:
        print("Booking Cancelled")
        return
    selected[1]["bookings"] += 1
    print(f"Movie ticket for [{selected[1]['title']}] has been booked.")
    return


### Sort Menu List
This block defines the menu options for sorting movies.

Each item in the list is a dictionary containing:

* `entry_name`: the text displayed in the menu
* `action`: the function that runs when the option is selected

The `lambda` functions are used because some menu actions need to call `sort_movies_list()` with arguments.

For example:

```python
lambda: sort_movies_list("title", desc=True)
```

sorts the movies by title in descending order.

In [ ]:
sort_menu_list = [
    {
        "entry_name": "Sort by Title (A to Z)",
        "action": lambda: sort_movies_list("title"),
    },
    {
        "entry_name": "Sort by Title (Z to A)",
        "action": lambda: sort_movies_list("title", desc=True),
    },
    {"entry_name": "Sort by Earliest", "action": lambda: sort_movies_list("date")},
    {
        "entry_name": "Sort by Latest",
        "action": lambda: sort_movies_list("date", desc=True),
    },
    {
        "entry_name": "Sort by Price (Lowest)",
        "action": lambda: sort_movies_list("price"),
    },
    {
        "entry_name": "Sort by Price (Highest)",
        "action": lambda: sort_movies_list("price", desc=True),
    },
]

### Admin Menu List
This block defines the admin menu.

The admin can add movies, remove movies, view all movies, and open the sort menu.

The sort option uses a lambda function because it needs to call `run_menu()` with the sort menu list and a title.

In [ ]:
admin_menu_list = [
    {"entry_name": "Add Movie", "action": add_movie},
    {"entry_name": "Remove Movie", "action": remove_movie},
    {"entry_name": "View All Movies", "action": view_all},
    {
        "entry_name": "Sort Movies",
        "action": lambda: run_menu(sort_menu_list, "Sort Movies"),
    },
]

### `admin_mode()` Function
This function controls access to admin features.

It first asks the user to enter the admin password. The entered password is compared with the value stored in `PASSWORD`. The default password is `Welcome@123`.

If the password is incorrect, an error message is displayed and the function stops.

If the password is correct, the admin menu is opened using `run_menu()`.

In [ ]:
def admin_mode():
    password_input = input("Enter Admin Password => ")
    if password_input != PASSWORD:
        print("Invalid Password.")
        return
    run_menu(admin_menu_list, "Admin Mode")

### User Menu List
```python
{
    "entry_name": "Sort by Title (A to Z)",
    "action": lambda: sort_movies_list("title"),
}
```

The menu lists store each menu option as a dictionary. Each dictionary has an `entry_name`, which is the text shown to the user, and an `action`, which stores the function to run when that option is selected.

Some actions use a **lambda function** because the function needs arguments.

For example:

```python
lambda: sort_movies_list("title")
```

This creates a small function that waits until the menu option is selected before calling `sort_movies_list("title")`.

This is necessary because writing:

```python
"action": sort_movies_list("title")
```

would call the function immediately when the menu list is created, instead of waiting for the user to choose that option.

So, the lambda delays the function call until the menu system runs:

```python
menu_list[choice - 1]["action"]()
```

In [ ]:
user_menu_list = [
    {
        "entry_name": "View All Movies",
        # view_all function is called with False as an argument to only
        # display movies which are available for booking
        "action": lambda: view_all(show_booked=False),
    },
    {"entry_name": "Book Movie Ticket", "action": book_ticket},
    {
        "entry_name": "Sort Movies",
        "action": lambda: run_menu(sort_menu_list, "Sort Movies"),
    },
]

### Main Menu List
This block defines the main menu of the application.

The user can choose between:

* Admin mode
* User mode

Admin mode calls the `admin_mode()` function. User mode opens the user menu using `run_menu()`.

In [ ]:
main_menu_list = [
    {"entry_name": "Admin", "action": admin_mode},
    {"entry_name": "User", "action": lambda: run_menu(user_menu_list, "User Mode")},
]

### `main()` Function
This function starts the program.

It opens the main menu by calling `run_menu()` and passing the main menu list. When the user exits the main menu, the program prints an exit message.

This function acts as the main controller of the application.

In [ ]:
def main():
    run_menu(main_menu_list, "Main Menu")
    print("\nExiting Application")


### Program Entry Point
This block checks whether the Python file is being run directly.

If the file is run directly, the `main()` function is called and the program starts.

This is a common Python practice because it prevents the program from automatically running if the file is imported into another Python file.

<div class="alert alert-info">
<b>Note:</b> To use Admin Mode, you can use the password <code>Welcome@123</code>
</div>


In [ ]:
if __name__ == "__main__":
    main()

---

## Conclusion

This program successfully implements a simple cinema booking system using Python. It separates different tasks into functions, making the code easier to understand and maintain. The use of helper functions improves input validation, while the reusable `run_menu()` function reduces repeated menu code.

The program also demonstrates important Python concepts such as lists, dictionaries, loops, conditionals, functions, lambda expressions, and date handling. Overall, it is a good example of a structured CLI application that manages movie data and allows both admin and user interaction.